In [1]:
from osgeo import gdal
import numpy as np


def read_raster(path):
    ds = gdal.Open(path)
    if ds is None:
        raise FileNotFoundError(f"Cannot open {path}")

    band = ds.GetRasterBand(1)
    arr = band.ReadAsArray().astype(np.float32)

    nodata = band.GetNoDataValue()
    if nodata is not None:
        arr[arr == nodata] = np.nan

    return ds, arr

def array_to_mem_ds(array, ref_ds):
    """
    Convert numpy array to in-memory GDAL dataset
    using reference dataset's georeference
    """
    driver = gdal.GetDriverByName("MEM")
    ds = driver.Create(
        "",
        ref_ds.RasterXSize,
        ref_ds.RasterYSize,
        1,
        gdal.GDT_Float32
    )

    ds.SetGeoTransform(ref_ds.GetGeoTransform())
    ds.SetProjection(ref_ds.GetProjection())

    band = ds.GetRasterBand(1)
    band.WriteArray(array)
    band.SetNoDataValue(np.nan)

    return ds


def delta_downscale_tmp(
    worldclim_path,
    era5_now_path,
    era5_clim_path,
    out_path
):
    """
    Delta downscaling for temperature (TMP)
    ERA5 instant: Kelvin
    ERA5 climatology & WorldClim: Celsius
    """

    # -----------------------------
    # 1. Read data
    # -----------------------------
    ds_wc, wc_arr = read_raster(worldclim_path)
    ds_now, now_arr = read_raster(era5_now_path)
    ds_clim, clim_arr = read_raster(era5_clim_path)

    # -----------------------------
    # 2. Unit conversion (K → ℃)
    # -----------------------------
    now_arr = now_arr - 273.15

    # -----------------------------
    # 3. Temperature anomaly
    # An_TMP(yr,m) = TMP(yr,m) − CRUClim_TMP(m)
    # -----------------------------
    an_tmp = now_arr - clim_arr
    an_tmp_ds = array_to_mem_ds(an_tmp, ds_now)

    # -----------------------------
    # 4. Resample anomaly to 1 km
    # -----------------------------
    mem_driver = gdal.GetDriverByName("MEM")
    an_tmp_1km_ds = mem_driver.Create(
        "",
        ds_wc.RasterXSize,
        ds_wc.RasterYSize,
        1,
        gdal.GDT_Float32
    )

    an_tmp_1km_ds.SetGeoTransform(ds_wc.GetGeoTransform())
    an_tmp_1km_ds.SetProjection(ds_wc.GetProjection())

    gdal.ReprojectImage(
        an_tmp_ds,
        an_tmp_1km_ds,
        an_tmp_ds.GetProjection(),
        ds_wc.GetProjection(),
        gdal.GRA_Bilinear
    )

    an_tmp_1km = an_tmp_1km_ds.GetRasterBand(1).ReadAsArray()

    # -----------------------------
    # 5. Delta correction
    # TMP(yr,m,res)
    # -----------------------------
    tmp_1km = wc_arr + an_tmp_1km

    # -----------------------------
    # 6. Write output
    # -----------------------------
    driver = gdal.GetDriverByName("GTiff")
    out_ds = driver.Create(
        out_path,
        ds_wc.RasterXSize,
        ds_wc.RasterYSize,
        1,
        gdal.GDT_Float32,
        options=["COMPRESS=LZW", "TILED=YES"]
    )

    out_ds.SetGeoTransform(ds_wc.GetGeoTransform())
    out_ds.SetProjection(ds_wc.GetProjection())

    band = out_ds.GetRasterBand(1)
    band.WriteArray(tmp_1km)
    band.SetNoDataValue(np.nan)

    out_ds.FlushCache()
    out_ds = None

    print(f"Finished: {out_path}")


In [2]:
if __name__ == "__main__":

    # ====== User inputs ======
    for yr in range(2000,2026):
        for m in range(1,13):
    # month (user-defined)
            worldclim = f"F:\\DF10\\WorldClim\\temperature\\wc2.1_30s_tavg_{m:02d}.tif"
            era5_clim = f"F:\\DF10\\ERA5_worldwide_mean\\ERA5_TEM_21base\\ERA5_TEM_1970_2000_{m}_mean_21base.tif"
            era5_now  = f"F:\\DF10\\ERA5_worldwide\\ERA5_TEM_21base\\ERA5_TEM_{yr}_{m}_21base.tif"
        
            out_file  = f"F:\\DF10\\Delta_DownScaled\\TMP_{yr}_{m}_1km_21base.tif"
        
            # ====== Run delta downscaling ======
            delta_downscale_tmp(
                worldclim_path=worldclim,
                era5_now_path=era5_now,
                era5_clim_path=era5_clim,
                out_path=out_file
            )

D:\Anaconda\Lib\site-packages\osgeo\gdal.py:330: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


Finished: F:\DF10\Delta_DownScaled\TMP_2000_1_1km_21base.tif
Finished: F:\DF10\Delta_DownScaled\TMP_2000_2_1km_21base.tif
Finished: F:\DF10\Delta_DownScaled\TMP_2000_3_1km_21base.tif
Finished: F:\DF10\Delta_DownScaled\TMP_2000_4_1km_21base.tif
Finished: F:\DF10\Delta_DownScaled\TMP_2000_5_1km_21base.tif
Finished: F:\DF10\Delta_DownScaled\TMP_2000_6_1km_21base.tif
Finished: F:\DF10\Delta_DownScaled\TMP_2000_7_1km_21base.tif
Finished: F:\DF10\Delta_DownScaled\TMP_2000_8_1km_21base.tif
Finished: F:\DF10\Delta_DownScaled\TMP_2000_9_1km_21base.tif
Finished: F:\DF10\Delta_DownScaled\TMP_2000_10_1km_21base.tif
Finished: F:\DF10\Delta_DownScaled\TMP_2000_11_1km_21base.tif
Finished: F:\DF10\Delta_DownScaled\TMP_2000_12_1km_21base.tif
Finished: F:\DF10\Delta_DownScaled\TMP_2001_1_1km_21base.tif
Finished: F:\DF10\Delta_DownScaled\TMP_2001_2_1km_21base.tif
Finished: F:\DF10\Delta_DownScaled\TMP_2001_3_1km_21base.tif
Finished: F:\DF10\Delta_DownScaled\TMP_2001_4_1km_21base.tif
Finished: F:\DF10\Del